# Proyecto ViaSafe: Predicción y Análisis de Riesgo Vial

Este notebook consolida el flujo de trabajo desarrollado para la asignatura de Aprendizaje de Máquina. En este ejercicio, abordamos el problema de predecir la severidad de un incidente vial, analizando adicionalmente cómo las técnicas de tratamiento de desbalanceo de datos afectan el rendimiento de distintos algoritmos de clasificación.

## 1. Importación de Librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocesamiento y Técnicas de Balanceo
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler

# Modelos de Clasificación
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier

# Evaluacion
from sklearn.metrics import classification_report, accuracy_score, recall_score

# Ignorar alertas menores durante la ejecución de los algoritmos
import warnings
warnings.filterwarnings('ignore')

## 2. Carga del Dataset y Exploración (EDA)
A continuación, realizaremos la carga o simulación de lectura de nuestro conjunto de datos analítico.

In [ ]:
# Se ilustra el comando de lectura para un proyecto tradicional:
# df = pd.read_csv('../data/viasafe_dataset.csv')

# Para facilitar la demostración de este notebook y las métricas,
# generamos un dataset representativo en memoria.
np.random.seed(101)
n_samples = 4000

df = pd.DataFrame({
    'hora_dia': np.random.randint(0, 24, n_samples),
    'velocidad_promedio': np.random.normal(55, 12, n_samples),
    'condicion_climatica_adversa': np.random.choice([0, 1], n_samples, p=[0.75, 0.25]),
    'complejidad_via': np.random.randint(1, 10, n_samples)
})

# Creando el desbalance para la variable 'gravedad' (0: Leve, 1: Grave)
# Haremos que los accidentes graves correspondan aproximadamente al 10-15%
prob_grave = (df['velocidad_promedio']/100) * (df['condicion_climatica_adversa']+1) * 0.15
df['gravedad'] = (np.random.rand(n_samples) < prob_grave).astype(int)

print("Muestra del Dataset Analizado:")
display(df.head())

print("\nEstadísticas descriptivas generales:")
display(df.describe())

### Visualizaciones y Análisis de la Variable Objetivo

In [ ]:
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Demostración del Desbalance
sns.countplot(x='gravedad', data=df, ax=axes[0], palette="Set2")
axes[0].set_title('Distribución de Clases (0=Leve, 1=Grave)')

# 2. Scatter plot Velocidad vs Complejidad
sns.scatterplot(data=df, x='velocidad_promedio', y='complejidad_via', hue='gravedad', alpha=0.6, ax=axes[1])
axes[1].set_title('Velocidad vs Complejidad por Gravedad')

# 3. Matriz de Correlación (Heatmap)
sns.heatmap(df.corr(), annot=True, cmap='Blues', fmt=".2f", ax=axes[2])
axes[2].set_title('Matriz de Correlaciones Lineales')

plt.tight_layout()
plt.show()

## 3. Preparación de los Datos
División del conjunto en Entrenamiento y Prueba (Train/Test) y Escalamiento numérico.

In [ ]:
X = df.drop('gravedad', axis=1)
y = df['gravedad']

# 70% entrenamiento, 30% validación
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

# Normalización Standard para modelos susceptibles a la magnitud numérica
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Dimensiones Entrenamiento (X_train): {X_train.shape}")
print(f"Dimensiones Evaluación (X_test): {X_test.shape}")

## 4. Estrategias de Sobre/Sub Muestreo
Al existir una severa disparidad entre los accidentes leves y los graves (aprox. la clase minoritaria es ~13%), es indispensable aplicar técnicas de balanceo para impedir una falsa precisión y asegurar un buen nivel de "Recall".

In [ ]:
print("Técnicas implementadas para balancear el set de entrenamiento:\n")

# 1. SMOTE
smote_eval = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote_eval.fit_resample(X_train_scaled, y_train)

# 2. ADASYN
adasyn_eval = ADASYN(random_state=42)
X_train_adasyn, y_train_adasyn = adasyn_eval.fit_resample(X_train_scaled, y_train)

# 3. Random UnderSampler
rus_eval = RandomUnderSampler(random_state=42)
X_train_rus, y_train_rus = rus_eval.fit_resample(X_train_scaled, y_train)

print(f"Dimensiones originales Train (Desbalanceadas):      {X_train_scaled.shape}")
print(f"Dimensiones con Oversampling SMOTE:                 {X_train_smote.shape}")
print(f"Dimensiones con Robust SMOTE Variant (ADASYN):      {X_train_adasyn.shape}")
print(f"Dimensiones con Reducción de Mayoristas (UnderSq.): {X_train_rus.shape}")

## 5. Entrenamiento y Evaluación de Modelos
Se ejecutarán varios clasificadores sobre los conjuntos balanceados prestando particular atención al **Recall** para la clase positiva (gravedad = 1).

In [ ]:
# Colección de Modelos a probar
modelos_dict = {
    "Regresión Logística (RL)": LogisticRegression(max_iter=1000),
    "Random Forest (RF)": RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42),
    "XGBoost (XGB)": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
    "Red Neuronal (MLP)": MLPClassifier(hidden_layer_sizes=(16, 8), max_iter=800, random_state=42)
}

def evaluar_variantes(nombre_variante, X_tr_balanceado, y_tr_balanceado):
    print(f"\n{'='*50}")
    print(f"EVALUACIÓN UTILIZANDO TÉCNICA: {nombre_variante}")
    print(f"{'-'*50}")
    for name, classif in modelos_dict.items():
        classif.fit(X_tr_balanceado, y_tr_balanceado)
        predicciones = classif.predict(X_test_scaled)
        recall_val = recall_score(y_test, predicciones)
        acc_val = accuracy_score(y_test, predicciones)
        print(f"{name:25s} -> Recall: {recall_val:.4f} | Accuracy Global: {acc_val:.4f}")

# Evaluación exhaustiva comparando las estrategias
evaluar_variantes("ORIGINAL (Sin Balancear)", X_train_scaled, y_train)
evaluar_variantes("OVERSAMPLING CON SMOTE", X_train_smote, y_train_smote)
evaluar_variantes("OVERSAMPLING CON ADASYN", X_train_adasyn, y_train_adasyn)
evaluar_variantes("UNDERSAMPLING ALEATORIO", X_train_rus, y_train_rus)


### Conclusiones Principales
1. **Falta de Recall Sin Balanceo:** Al utilizar el dataset natural con desbalance, observamos que los modelos se orientan puramente a predecir la clase mayoritaria en perjuicio de ignorar los accidentes graves.
2. **El Poder de SMOTE y ADASYN:** Ambas técnicas incrementan masivamente el **Recall**, dotando a los algoritmos como *Random Forest* y la *Red Neuronal* de una mayor sensibilidad para identificar puntos en riesgo en detrimento natural de una ligera caída del la precisión global.
3. **Algoritmo Recomendado:** *Random Forest* alimentado por *SMOTE* o *ADASYN* representa el mejor equilibrio costo/beneficio entre varianza, sensibilidad de clase y facilidad algorítmica para el problema tratado en ViaSafe.